# 📝Data exploration + Design justification document.

>This document was created in order to detail any analysis of the NASA dataset used to justify tables being created in relation to planetary data/observational data within the PostgreSQL database (OLTP).

## 🗺️ Data exploration

For this project atmospheric data from NASA's Exoplanet Archive was retrived using an endpoint of NASA's Table Access Protocol (TAP) service. The syncronous TAP endpoint was accesd to query Atmospheric Spectroscopy data and retrieve the results for analysis with python (Pandas).

### 📖Data Background

The Dataset cointains key information about exoplanet atmosphereric observations, including the spectroscopy method used, the instrument used, facility information, wavelength ranges and more.

Using NASA's API service, 1000+ records were programmatically retrieved in csv format including both numerical and categorical fields, as well as NULL values.

NASA Exoplanet Archive. Atmospheric Spectroscopy. California Institute of Technology. (accessed 8 September 2026)

Source: https://exoplanetarchive.ipac.caltech.edu/

NASA Exoplanet Archive DOI: 10.26133/NEA1

In [209]:
import requests
import pandas as pd
from io import StringIO
import numpy as np 

url = "https://exoplanetarchive.ipac.caltech.edu/TAP/sync"

params = {
    "query": """
        SELECT *
        FROM spectra
    """,
    "format": "csv"
}

response = requests.get(url, params=params) # Response from NASA's API endpoint
print(f"HTTP status_code: {response.status_code}\n") # Status code for validation
response.raise_for_status() # Error Handling 
df = pd.read_csv(StringIO(response.text)) # Treats text string as a file

HTTP status_code: 200



In [202]:
df.set_index(keys=np.arange(len(df)))
df.index.name = 'id'
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1826 entries, 0 to 1825
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   pl_name         1826 non-null   object 
 1   spec_type       1826 non-null   object 
 2   authors         1826 non-null   object 
 3   num_datapoints  1826 non-null   int64  
 4   instrument      1818 non-null   object 
 5   facility        1826 non-null   object 
 6   minwavelng      1826 non-null   float64
 7   maxwavelng      1826 non-null   float64
 8   mintranmid      915 non-null    float64
 9   maxtranmid      915 non-null    float64
 10  note            1414 non-null   object 
 11  bibcode         1826 non-null   object 
 12  spec_path       1826 non-null   object 
dtypes: float64(4), int64(1), object(8)
memory usage: 185.6+ KB


The table above provided by `.info()` gives some insight into the dataset and amount of records in the dataset.

The table below gives details about each field within the table. 
 
| Table Label and Unit | Database Column Name | Description |
|---|---|---|
| Planet Name† | `pl_name` | Planetary name most commonly used in the literature. |
| Type of Spectrum† | `spec_type` | Specifies the type of spectral data in that row:<br>• Transmission (taken during transit)<br>• Eclipse (taken during secondary eclipse)<br>• Direct Imaging (taken of a directly imaged exoplanet) |
| Bibcode | `bibcode` | The bibliographic code that identifies the reference paper in the Astrophysics Data System (ADS). |
| Reference† | `authors` | Link to the refereed publication describing the observation. |
| # Datapoints† | `num_datapoints` | The number of data points in the spectrum. |
| Instrument† | `instrument` | Name of the instrument used to perform the observation. |
| Facility† | `facility` | Name of the facility/telescope used to perform the observation. |
| Minimum Wavelength [microns]† | `minwavelng` | The minimum wavelength value from the spectrum file. |
| Maximum Wavelength [microns]† | `maxwavelng` | The maximum wavelength value from the spectrum file. |
| Min. Obs. Date [BJD]† | `mintrannmid` | If the spectrum is from a single epoch, the date of observation—either transit/eclipse midpoint, or time of observation.<br><br>If the spectrum is combined from multiple transits, the first (smallest) date of observation. |
| Max. Obs. Date [BJD]† | `maxtrannmid` | If the spectrum is from a single epoch, the date of observation—either transit/eclipse midpoint, or time of observation.<br><br>If the spectrum is combined from multiple transits, the last (largest) date of observation. |
| Notes† | `note` | Additional information about the spectrum (e.g., the specific table in the reference paper where data were taken). |
| File Path | `spec_path` | The spectrum's file name, which can be used to map downloaded files to the metadata. |

NASA Exoplanet Archive (n.d.) Atmospheres Columns. 
Available at: NASA Exoplanet Archive – Atmospheres Columns (Accessed: 8 September 2026).



### 🛠️ Null counts

In [203]:
# Finding Nulls
print("---col null counts---")
df.isna().sum()

---col null counts---


pl_name             0
spec_type           0
authors             0
num_datapoints      0
instrument          8
facility            0
minwavelng          0
maxwavelng          0
mintranmid        911
maxtranmid        911
note              412
bibcode             0
spec_path           0
dtype: int64

The above series provides counts for `NULL` values within each column. instrument, mintranmid, maxtranmid and note columns in particular contain `NULL` values with the max amount of nulls for a column being 911.

For the PostgreSQL database, keeping these values as `NULL` is fine as potential updates to the data may have these values filled.

Since majority of the columns contain 0 null values it can be justified to set `NOT NULL` constraints on some of these columns such as pl_name or spec_path.

### 📃 Atmospheric Spectra dataset preview

In [ ]:
df.head(5)

---First 5 records---


,pl_name,spec_type,authors,num_datapoints,instrument,facility,minwavelng,maxwavelng,mintranmid,maxtranmid,note,bibcode,spec_path
id,,,,,,,,,,,,,
0,Kepler-20 c,Transmission,D&eacute;sert et al. 2015,1,Infrared Array Camera (IRAC),Spitzer Space Telescope satellite,4.5000,4.5000,NaN,NaN,NaN,2015ApJ...804...59D,56/12/36/06/Kepler_20_c_3.101_3665_1.tbl
1,55 Cnc e,Transmission,de Mooij et al. 2014,8,ALFOSC,Nordic Optical Telescope,0.4735,0.6560,NaN,NaN,NaN,2014ApJ...797L..21D,78/51/78/87/55_Cnc_e_3.10924_3673_1.tbl
2,55 Cnc e,Transmission,Tsiaras et al. 2016,25,Wide Field Camera 3,Hubble Space Telescope satellite,1.1320,1.6376,NaN,NaN,NaN,2016ApJ...820...99T,18/43/09/97/55_Cnc_e_3.10924_3750_1.tbl
3,CoRoT-1 b,Eclipse,Gillon et al. 2009,1,High Acuity Wide-field K-band Imager (HAWK-I),European Southern Observatory (ESO) 8.2m Very ...,2.0950,2.0950,NaN,NaN,Table 1,2009A&A...506..359G,52/88/66/83/CoRoT_1_b_3.10951_2058_1.tbl
4,CoRoT-1 b,Transmission,Schlawin et al. 2014,10,SpeX and MORIS,Infrared Telescope Facility,0.8600,2.3120,NaN,NaN,NaN,2014ApJ...783....5S,30/21/22/41/CoRoT_1_b_3.10951_3421_1.tbl


### 🙍‍♂️🙍‍♂️Duplicate check

In [213]:
# Check for duplicate records
print(f'There are {df.duplicated().sum()} duplicate records in this table.\n')
print('---Duplicate values in column count---\n')
for column in list(df.columns):
    print(f'{column} duplicate counts: {df[column].duplicated().sum()}')
print('--------------------------------------')

There are 0 duplicate records in this table.

---Duplicate values in column count---

pl_name duplicate counts: 1537
spec_type duplicate counts: 1823
authors duplicate counts: 1374
num_datapoints duplicate counts: 1672
instrument duplicate counts: 1698
facility duplicate counts: 1736
minwavelng duplicate counts: 1314
maxwavelng duplicate counts: 1328
mintranmid duplicate counts: 1013
maxtranmid duplicate counts: 1012
note duplicate counts: 829
bibcode duplicate counts: 1334
spec_path duplicate counts: 0
--------------------------------------


The above information shows that each record within the dataset is unique which means cleaning the data by removing duplicate data would be unnecessary. Additionally since there are no `NULL` values key columns such as pl_name, spec_type, authors, bibcode and spec_path removing records is not required. 

Since **spec_path** has no duplicate values it enforces uniqueness within the dataset and thus could do the same within an **atmospheric_spectra** table within the PostgreSQL database.

### 🛠️ Unique values

In [208]:
#Unique value counts
for column in list(df.columns):
    print(f'------{column.upper()} unique values count------\n')
    print(df[column].value_counts())
    print('')
df.to_csv("backup.csv")

------PL_NAME unique values count------

pl_name
HD 189733 b       64
HD 209458 b       47
WASP-43 b         40
WASP-19 b         37
WASP-12 b         36
                  ..
HR 2562 b          1
WD 0806-661 b      1
TOI-431 b          1
HD 143811 AB b     1
Kepler-45 b        1
Name: count, Length: 289, dtype: int64

------SPEC_TYPE unique values count------

spec_type
Transmission      940
Eclipse           801
Direct Imaging     85
Name: count, dtype: int64

------AUTHORS unique values count------

authors
Deming et al. 2023           449
D&eacute;sert et al. 2015     72
Changeat et al. 2022          66
Garhart et al. 2020           40
Baxter et al. 2021            32
                            ... 
Todorov et al. 2010            1
Buhler et al. 2016             1
Knutson et al. 2011            1
Rackham et al. 2017            1
Diamond-Lowe et al. 2023       1
Name: count, Length: 452, dtype: int64

------NUM_DATAPOINTS unique values count------

num_datapoints
1       687
2      

The above code provides more insight into the number unique values in each column. This step is crutial since it will infom which fields are likely to form their own **look up tables, fact and dimension tables**.

Upon inspecting the data above, it was decided that the **SPEC_TYPE** field should form it's own look up table since it contains only 3 unique types of spectral data that are repeatedly used by throughout the dataset. 

The **author** and **bibcode** columns share similar counts for unique values and thus warrent futher inspection.

In [ ]:
for column in ["authors", "bibcode"]:
    print(f'------{column} unique values------\n')
    print(df[column].value_counts())
    print('')

------authors unique values------

authors
Deming et al. 2023           449
D&eacute;sert et al. 2015     72
Changeat et al. 2022          66
Garhart et al. 2020           40
Baxter et al. 2021            32
                            ... 
Todorov et al. 2010            1
Buhler et al. 2016             1
Knutson et al. 2011            1
Rackham et al. 2017            1
Diamond-Lowe et al. 2023       1
Name: count, Length: 452, dtype: int64

------bibcode unique values------

bibcode
2023AJ....165..104D    449
2015ApJ...804...59D     72
2022ApJS..260....3C     66
2020AJ....159..137G     40
2021A&A...648A.127B     32
                      ... 
2010ApJ...708..498T      1
2015A&A...580A..60M      1
2016ApJ...821...26B      1
2011ApJ...735...27K      1
2022A&A...658A.133G      1
Name: count, Length: 492, dtype: int64



By taking a glance at the data above it is not unreasonable to assume that each bibcode has one author (or set of authors) however, there are 452 values in the authors series and 492 in the bibcode series. This indicates that some records can share bibcodes and authors. Finding unique author and bibcode combinations would be benificial if a **publication table** is to be created within postgrSQL.  

In [211]:
print(str(len(df[df[['bibcode','authors']].duplicated()][['bibcode','authors']].value_counts())) + " unique bibcode-author combos") 
df[df[['bibcode','authors']].duplicated()][['bibcode','authors']].value_counts()

243 unique bibcode-author combos


bibcode              authors                                 
2023AJ....165..104D  Deming et al. 2023                          448
2015ApJ...804...59D  D&eacute;sert et al. 2015                    71
2022ApJS..260....3C  Changeat et al. 2022                         65
2020AJ....159..137G  Garhart et al. 2020                          39
2021A&A...648A.127B  Baxter et al. 2021                           31
                                                                ... 
2021AJ....161...44E  Edwards et al. 2021                           1
2024AJ....168...51K  Kammerer et al. 2024                          1
2013Icar..225..432S  Swain et al. 2013                             1
2024AJ....167..214P  P&eacute;rez-Gonz&aacute;lez et al. 2024      1
2023Natur.620...67K  Kempton et al. 2023                           1
Name: count, Length: 243, dtype: int64

The code written above has located all 243 of the unique bibcode-author combinations within this data set. These unique combinations will be used to create a **publication look up table** within the postgeSQL database with the ability to be updated, so long as the new entry is a unique author-bibcode combination. 

## 📊 Table design justification
For brevity, the next set of tables to be included within the postgreSQL database are listed below with justifications given.

| Table name | Justification |
|---|---|
| Planet|This table is justified since multiple spectral files can be about the same planet and thus would reference said planets within a shared table. Making this a look up table would ensure data integrity by reducing duplicates. This table would contain only the unique **pl_name**'s from this dataset.|
|Publication| This table is justified since multiple publications can reference the same planets, facilities and instruments. This table leads to the reduction of duplicate values. This table would contain the unique **bibcode** and **authors**  combinations from this dataset.|
|Facility| The justification for this table is clear since multiple observations take place in different facilities and are conducted by different researchers. This table would contain only unique facility colum data.|
|Instrument| An instrument table is justified since many different spectra observations use the same equipment for different planets. This table would only contain unique instruments from the instruments column.|
|Atmospheric_spectra|This table would act as the 'main' or fact table containing data minwavelng, maxwavelng, mintranmid, maxtranmid, note, num_datapoints and spec_path field data. This table would contain foreign keys that reference the tables previously mentioned. It may contain duplicate data in the listed fields however uniqueness is inforced by the spec_path since they are all distinct.|